In [ ]:
import pandas as pd

calendar = pd.read_csv('Data/calendar.csv')
items = pd.read_csv('Data/items_per_season.csv')
cost_amount = pd.read_csv('Data/cost_amount.csv')
mail_hashed_all = pd.read_csv('Data/mail_hashed_all.csv')
customers_fy_16_17 = pd.read_csv('Data/customers_fy_16_17.csv')
customers_fy_23_24 = pd.read_csv('Data/customers_fy_23_24.csv')
items_2016 = pd.read_csv('Data/items_2016.csv')
items_2022 = pd.read_csv('Data/items_2022.csv')
orders_fy_16_17 = pd.read_csv('Data/order_invoice_refunds_fy_16_17.csv')
orders_fy_22_23 = pd.read_csv('Data/order_invoice_refunds_fy_22_23.csv')

In [ ]:
# Kunden zusammenführen & H_ID anlegen
customers_base = pd.concat([customers_fy_16_17, customers_fy_23_24], ignore_index=True)
customers_base = customers_base.drop_duplicates().reset_index(drop=True)

customers_all_hid = customers_base.merge(mail_hashed_all, on='CUSTOMER_NO', how='left')
customers_all_hid = customers_all_hid.rename(columns={'EMAIL_HASHED': 'H_ID'})

print(f"Kunden gesamt (dedup): {len(customers_base):,}")
print(f"Kunden mit H_ID:       {customers_all_hid['H_ID'].notna().sum():,}")
print(f"Unique H_IDs:          {customers_all_hid['H_ID'].nunique():,}")
display(customers_all_hid.head())


In [ ]:
#Mapping + Kundenprofil pro H_ID
cust_hid_map = (
    customers_all_hid[['CUSTOMER_NO', 'H_ID']]
    .drop_duplicates(subset='CUSTOMER_NO')
)

customers_hid_profile = (
    customers_all_hid
    .drop_duplicates(subset='H_ID')
    .reset_index(drop=True)
)

print(f"Mapping-Einträge (CUSTOMER_NO → H_ID): {len(cust_hid_map):,}")
print(f"Kundenprofile pro H_ID:                {len(customers_hid_profile):,}")
print(f"Spalten im Kundenprofil:               {list(customers_hid_profile.columns)}")


In [ ]:
#Items kombinieren
items_all_cols = (
    pd.concat([items_2016, items_2022], ignore_index=True)
    .drop_duplicates(subset='ITEM_NO')
    .reset_index(drop=True)
)

print(f"Items gesamt (dedup): {len(items_all_cols):,}")
print(f"Spalten:              {list(items_all_cols.columns)}")
display(items_all_cols.head())


In [ ]:
#Orders kombinieren, H_ID + Items + Kosten joinen
orders_base = pd.concat([orders_fy_16_17, orders_fy_22_23], ignore_index=True)
orders_base = orders_base.drop_duplicates().reset_index(drop=True)
orders_base['ORDER_DATE'] = pd.to_datetime(orders_base['ORDER_DATE'], errors='coerce')

orders_enriched_hid = (
    orders_base
    .merge(cust_hid_map,   on='CUSTOMER_NO',              how='left')
    .merge(items_all_cols, on='ITEM_NO',                  how='left', suffixes=('', '_item'))
    .merge(cost_amount,    on=['DOCUMENT_NO', 'LINE_NO'],  how='left')
)

print(f"Order-Zeilen gesamt:        {len(orders_enriched_hid):,}")
print(f"Davon ohne H_ID (kein Mail): {orders_enriched_hid['H_ID'].isna().sum():,}")
print(f"Spalten gesamt:              {len(orders_enriched_hid.columns)}")
display(orders_enriched_hid.head())


In [ ]:
#Aggregation auf Order-Ebene pro H_ID
order_level_hid = (
    orders_enriched_hid
    .groupby(['H_ID', 'ORDER_NO', 'ORDER_DATE'])
    .agg(
        net_amount_order  = ('NET_AMOUNT',           'sum'),
        cost_amount_order = ('COST_AMOUNT',          'sum'),
        qty_order         = ('QUANTITY',             'sum'),
        n_line_items      = ('ITEM_NO',              'count'),
        n_unique_items    = ('ITEM_NO',              'nunique'),
        n_customer_nos    = ('CUSTOMER_NO',          'nunique'),
        has_discount      = ('LINE_DISCOUNT_AMOUNT', lambda x: (x > 0).any()),
        discount_code     = ('DISCOUNT_CODE',        'first'),
        country_code      = ('COUNTRY_CODE',         'first'),
        posting_date      = ('POSTING_DATE',         'first'),
        season_order_date = ('SEASON_ORDER_DATE',    'first'),
        season_post_date  = ('SEASON_POSTING_DATE',  'first'),
        uvp_sum           = ('UVP',                  'sum'),
        gross_amount      = ('SALES_PRICE_ORIGINAL', 'sum'),
    )
    .reset_index()
)
order_level_hid['is_return_order'] = order_level_hid['qty_order'] <= 0

print(f"Bestellungen gesamt:   {len(order_level_hid):,}")
print(f"Retouren:              {order_level_hid['is_return_order'].sum():,}")
print(f"H_IDs mit >1 CUST_NO: {(order_level_hid['n_customer_nos'] > 1).sum():,}")
display(order_level_hid.head())




In [ ]:
#Panel: H_ID × Order mit vollem Kundenprofil
panel_hid = (
    order_level_hid
    .merge(customers_hid_profile, on='H_ID', how='left')
    .sort_values(['H_ID', 'ORDER_DATE'])
    .reset_index(drop=True)
)

panel_hid['order_rank'] = (
    panel_hid.groupby('H_ID')['ORDER_DATE']
    .rank(method='dense').astype(int)
)

print(f"Panel (H_ID-basiert): {panel_hid['H_ID'].nunique():,} Kunden | {len(panel_hid):,} Bestellungen")
print(f"Davon Retouren:       {panel_hid['is_return_order'].sum():,}")
print(f"Spalten gesamt:       {len(panel_hid.columns)}")
display(panel_hid.head(10))


In [ ]:
# Mapping H_ID → kurze numerische ID (1, 2, 3, ...)
hid_to_int = {hid: i for i, hid in enumerate(panel_hid['H_ID'].dropna().unique(), start=1)}

panel_hid.insert(
    panel_hid.columns.get_loc('H_ID') + 1,  # direkt nach H_ID einfügen
    'CUSTOMER_ID',
    panel_hid['H_ID'].map(hid_to_int)
)

print(f"Unique CUSTOMER_IDs: {panel_hid['CUSTOMER_ID'].nunique():,}")
print(f"Range: {panel_hid['CUSTOMER_ID'].min()} – {panel_hid['CUSTOMER_ID'].max()}")
display(panel_hid[['CUSTOMER_ID', 'H_ID', 'ORDER_NO', 'ORDER_DATE']].head(10))


In [ ]:
# EMAIL_HASH = langer Hash, H_ID = kurze Zahl ab 1
panel_hid = panel_hid.rename(columns={'H_ID': 'EMAIL_HASH'})

hid_to_int = {
    hid: i
    for i, hid in enumerate(
        panel_hid['EMAIL_HASH'].dropna().unique(), start=1
    )
}

panel_hid.insert(
    panel_hid.columns.get_loc('EMAIL_HASH') + 1,
    'H_ID',
    panel_hid['EMAIL_HASH'].map(hid_to_int).astype('Int64')
)

print(f"Unique H_IDs: {panel_hid['H_ID'].nunique():,}")
print(f"Range: {panel_hid['H_ID'].min()} – {panel_hid['H_ID'].max()}")
display(
    panel_hid[['H_ID', 'EMAIL_HASH', 'CUSTOMER_NO', 'ORDER_NO', 'ORDER_DATE', 'order_rank']]
    .head(10)
)


In [ ]:
# Panel neu aufbauen + H_ID (kurze Zahl) + EMAIL_HASH (ganz rechts)
panel_hid = (
    order_level_hid
    .merge(customers_hid_profile, on='H_ID', how='left')
    .sort_values(['H_ID', 'ORDER_DATE'])
    .reset_index(drop=True)
)

panel_hid['order_rank'] = (
    panel_hid.groupby('H_ID')['ORDER_DATE']
    .rank(method='dense').astype(int)
)

# Hash umbenennen
panel_hid = panel_hid.rename(columns={'H_ID': 'EMAIL_HASH'})

# Kurze numerische H_ID ab 1
hid_to_int = {
    hid: i
    for i, hid in enumerate(panel_hid['EMAIL_HASH'].dropna().unique(), start=1)
}
panel_hid.insert(0, 'H_ID', panel_hid['EMAIL_HASH'].map(hid_to_int).astype('Int64'))

# EMAIL_HASH ganz nach rechts
panel_hid = panel_hid[[c for c in panel_hid.columns if c != 'EMAIL_HASH'] + ['EMAIL_HASH']]

print(f"Panel (H_ID-basiert): {panel_hid['H_ID'].nunique():,} Kunden | {len(panel_hid):,} Bestellungen")
print(f"Davon Retouren:       {panel_hid['is_return_order'].sum():,}")
print(f"Spalten gesamt:       {len(panel_hid.columns)}")
display(
    panel_hid[['H_ID', 'CUSTOMER_NO', 'ORDER_NO', 'ORDER_DATE', 'order_rank', 'EMAIL_HASH']]
    .head(10)
)


In [ ]:
panel_hid.to_csv('Processed Data/big_panel.csv', index=False)

print(f"Gespeichert: 'Processed Data/big_panel.csv'")
print(f"Zeilen: {len(panel_hid):,} | Spalten: {len(panel_hid.columns)}")


Merge PLZ Income and Kids statistics

In [ ]:
kids_raw      = pd.read_csv('Data/kids_per_bundesland.csv')
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Bevoelkerung/Haushalte-Familien/Tabellen/2-7-familien-bundeslaender.html
income_raw    = pd.read_csv('Data/income_per_city_region_2023.csv')
#https://www.statistikportal.de/de/vgrdl/ergebnisse-kreisebene/einkommen-kreise
#Einkommen der priaten Haushalte
plz_mapping   = pd.read_csv('Data/mapping_postcode.csv')
#https://gist.github.com/pmdroid/6ae8286a494cafce82b6ea5f6cc2362a#file-german-postcodes-csv

In [ ]:
#Quelle
kids_raw = pd.read_csv('Data/kids_per_bundesland.csv')

aggregat_zeilen = ['Deutschland', 'Westdeutschland ohne Berlin', 'Ostdeutschland einschließlich Berlin']
kids = kids_raw[~kids_raw['Bundesland'].isin(aggregat_zeilen)].reset_index(drop=True)

print(f"Bundesländer: {len(kids)}")
display(kids)


In [ ]:
income_raw = pd.read_csv('Data/income_per_city_region_2023.csv')

print(f"Zeilen: {len(income_raw):,} | nuts_level-Verteilung:")
print(income_raw['nuts_level'].value_counts().sort_index())
display(income_raw.head())


In [ ]:
income_city = (
    income_raw[income_raw['nuts_level'] == 3]
    [['gebietseinheit', 'land', 'regionalschluessel', 'y2023']]
    .copy()
    .rename(columns={'y2023': 'kaufkraft_2023'})
)

# Kernname: alles vor erstem Komma, Suffixe entfernen, lowercase
income_city['city_clean'] = (
    income_city['gebietseinheit']
    .str.split(',').str[0]
    .str.strip()
    .str.replace(r'\s+(Landkreis|Stadtkreis|Kreis|Stadt|Regierungsbezirk)$', '', regex=True)
    .str.strip()
    .str.lower()
)

# Bei Duplikaten (z.B. Karlsruhe Stadtkreis + Landkreis): Stadtkreis bevorzugen
income_city['_stadtkreis'] = income_city['gebietseinheit'].str.contains('Stadtkreis', case=False)
income_city = (
    income_city
    .sort_values('_stadtkreis', ascending=False)
    .drop_duplicates(subset='city_clean')
    .drop(columns='_stadtkreis')
    .reset_index(drop=True)
)

print(f"Income-Einträge (nuts=3, dedup): {len(income_city):,}")
display(income_city[['gebietseinheit', 'city_clean', 'kaufkraft_2023']].head(10))


In [ ]:
customers_city = (
    customers_all_hid[customers_all_hid['COUNTRY_REGION_CODE'] == 'DE']
    [['POST_CODE', 'CITY']].drop_duplicates().dropna(subset=['CITY'])
)
customers_city['POST_CODE'] = customers_city['POST_CODE'].astype(str).str.zfill(5)

customers_city['city_clean'] = (
    customers_city['CITY']
    .str.strip()
    .str.replace(r'\s*/\s*', ' ', regex=True)   # "Halle / Saale" → "Halle  Saale"
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)

print(f"Unique PLZ/Ort-Kombinationen: {len(customers_city):,}")
display(customers_city.head(10))


In [ ]:
plz_mapping = pd.read_csv('Data/mapping_postcode.csv')
plz_mapping['Plz'] = plz_mapping['Plz'].astype(str).str.zfill(5)

print(f"Einträge: {len(plz_mapping):,} | Unique PLZs: {plz_mapping['Plz'].nunique():,}")
display(plz_mapping.head())


In [ ]:
merged_bl = (
    customers_city
    .merge(
        plz_mapping[['Plz', 'Bundesland']].drop_duplicates(subset='Plz'),
        left_on='POST_CODE', right_on='Plz',
        how='left'
    )
    .drop(columns='Plz')
    .merge(kids, on='Bundesland', how='left')
)

print(f"Zeilen:                {len(merged_bl):,}")
print(f"Ohne Bundesland-Match: {merged_bl['Bundesland'].isna().sum():,}")
display(merged_bl.head(10))


In [ ]:
# Cleaning: Ort aus plz_mapping & CITY aus customers
merged_bl['city_clean'] = merged_bl['CITY'].str.strip().str.lower()

income_city['city_clean'] = (
    income_raw[income_raw['nuts_level'] == 3]['gebietseinheit']
    .str.split(',').str[0]
    .str.replace(r'\s+(Landkreis|Stadtkreis|Kreis|Stadt)$', '', regex=True)
    .str.strip()
    .str.lower()
)

# Check: welche Kundenstädte matchen NICHT?
unmatched_cities = set(merged_bl['city_clean']) - set(income_city['city_clean'])
print(f"Städte ohne Match: {len(unmatched_cities):,} von {merged_bl['city_clean'].nunique():,}")
print("Beispiele:", sorted(unmatched_cities)[:20])


In [ ]:
income_clean = (
    income_raw[income_raw['nuts_level'] == 3]
    [['gebietseinheit', 'y2023']]
    .copy()
    .rename(columns={'y2023': 'kaufkraft_2023'})
)
income_clean['city_clean'] = (
    income_clean['gebietseinheit']
    .str.split(',').str[0]
    .str.replace(r'\s+(Landkreis|Stadtkreis|Kreis|Stadt)$', '', regex=True)
    .str.strip()
    .str.lower()
)
income_clean = income_clean.drop_duplicates(subset='city_clean')

plz_stats = merged_bl.merge(income_clean, on='city_clean', how='left')

matched   = plz_stats['kaufkraft_2023'].notna().sum()
unmatched = plz_stats['kaufkraft_2023'].isna().sum()
print(f"Gesamt:        {len(plz_stats):,}")
print(f"Mit Ort-Match: {matched:,}  ({matched/len(plz_stats)*100:.1f}%)")
print(f"Ohne Match:    {unmatched:,}")
display(plz_stats.head(10))


second try 


In [ ]:
# mapping_postcode: saubere Ort-Namen pro PLZ
plz_mapping['Plz'] = plz_mapping['Plz'].astype(str).str.zfill(5)
plz_mapping['city_clean'] = plz_mapping['Ort'].str.strip().str.lower()

# income: nur nuts_level=3 (Landkreis-Ebene, granularst verfügbar)
income_nuts3 = (
    income_raw[income_raw['nuts_level'] == 3]
    [['gebietseinheit', 'regionalschluessel', 'land', 'y2023']]
    .copy()
    .rename(columns={'y2023': 'kaufkraft_2023'})
)
income_nuts3['city_clean'] = (
    income_nuts3['gebietseinheit']
    .str.split(',').str[0]
    .str.strip()
    .str.lower()
)
# Bei Duplikaten: Stadtkreis bevorzugen (präziser als Landkreis)
income_nuts3['_prio'] = income_nuts3['gebietseinheit'].str.contains(
    'Stadtkreis|Universitätsstadt|Landeshauptstadt|Freie.*Stadt', case=False, regex=True
)
income_nuts3 = (
    income_nuts3.sort_values('_prio', ascending=False)
    .drop_duplicates(subset='city_clean')
    .drop(columns='_prio')
    .reset_index(drop=True)
)

print(f"income-Einträge (nuts=3, dedup): {len(income_nuts3):,}")
display(income_nuts3[['gebietseinheit', 'city_clean', 'kaufkraft_2023']].head(10))


In [ ]:
# Unique PLZs aus Kundendaten (nur DE)
unique_plzs = (
    customers_all_hid[customers_all_hid['COUNTRY_REGION_CODE'] == 'DE']
    [['POST_CODE']].drop_duplicates().dropna()
)
unique_plzs['POST_CODE'] = unique_plzs['POST_CODE'].astype(str).str.zfill(5)

# PLZ → sauberer Ort + Bundesland (aus mapping_postcode, nicht aus schmutzigem CITY!)
plz_enriched = unique_plzs.merge(
    plz_mapping[['Plz', 'Ort', 'Bundesland', 'city_clean']].drop_duplicates(subset='Plz'),
    left_on='POST_CODE', right_on='Plz', how='left'
).drop(columns='Plz')

# Left join kids via Bundesland
plz_enriched = plz_enriched.merge(kids, on='Bundesland', how='left')

print(f"Unique PLZs:           {len(plz_enriched):,}")
print(f"Ohne Bundesland-Match: {plz_enriched['Bundesland'].isna().sum():,}")
display(plz_enriched.head(10))


In [ ]:
plz_stats = plz_enriched.merge(
    income_nuts3[['city_clean', 'gebietseinheit', 'regionalschluessel', 'kaufkraft_2023']],
    on='city_clean', how='left'
)

matched   = plz_stats['kaufkraft_2023'].notna().sum()
unmatched = plz_stats['kaufkraft_2023'].isna().sum()
print(f"Gesamt:                {len(plz_stats):,}")
print(f"Mit Landkreis-Match:   {matched:,}  ({matched/len(plz_stats)*100:.1f}%)")
print(f"Ohne Match (nur BL):   {unmatched:,}")

# Ungematchte Orte zeigen
print("\nTop ungematchte Orte:")
print(plz_stats[plz_stats['kaufkraft_2023'].isna()]['Ort'].value_counts().head(15))
display(plz_stats.head(10))
